# BetaEarth: AlphaEarth Embedding Emulator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asterisk-labs/beta-earth/blob/main/examples/demo.ipynb)
[![PyPI](https://img.shields.io/pypi/v/betaearth)](https://pypi.org/project/betaearth/)
[![HuggingFace](https://img.shields.io/badge/%F0%9F%A4%97-Models-yellow)](https://huggingface.co/asterisk-labs)

This notebook shows how to predict dense 10m geospatial embeddings from Sentinel-2 imagery using BetaEarth.

**What you'll do:**
1. Install BetaEarth and load the best model
2. Download a real Sentinel-2 tile from Major TOM
3. Predict embeddings and visualise as PCA-RGB
4. Compare with real AlphaEarth embeddings
5. Explore model variants

## 1. Install

In [ ]:
!pip install -q betaearth rasterio matplotlib scikit-learn s3fs geopandas shapely major-tom-grid

## 2. Load model

The primary model is a frozen SegFormer-B2 with FiLM temporal conditioning — **0.886 cosine similarity** with AlphaEarth, using only **0.3M trainable parameters**.

In [ ]:
from betaearth import BetaEarth

model = BetaEarth.from_pretrained("asterisk-labs/betaearth-segformer-film")
print(model)

## 3. Download a sample tile from Major TOM

We fetch a single Sentinel-2 L2A tile directly from its parquet shard on HuggingFace — no need to download the full dataset.

In [ ]:
import io
import numpy as np
import pandas as pd
import rasterio
from huggingface_hub import hf_hub_download

TARGET_CELL = "283U_505R"

# Step 1: Download the metadata index (small) to find the parquet shard
meta_path = hf_hub_download("Major-TOM/Core-S2L2A", "metadata.parquet", repo_type="dataset")
meta = pd.read_parquet(meta_path)
row = meta[meta.grid_cell == TARGET_CELL].iloc[0]

print(f"Target: {TARGET_CELL}")
print(f"Product: {row.product_id}")
print(f"Parquet shard: {row.parquet_url.split('/')[-1]}, row {row.parquet_row}")

In [ ]:
# Step 2: Download just the parquet shard and read one row
shard_file = row.parquet_url.split("/")[-1]
shard_path = hf_hub_download("Major-TOM/Core-S2L2A", f"images/{shard_file}", repo_type="dataset")
shard = pd.read_parquet(shard_path)
tile = shard.iloc[row.parquet_row]

def decode_band(band_bytes):
    """Decode a GeoTIFF band from bytes."""
    with rasterio.open(io.BytesIO(band_bytes)) as src:
        return src.read(1)

# Stack the 9 bands BetaEarth needs: [B02,B03,B04,B08,B05,B06,B07,B11,B12]
band_names_10m = ["B02", "B03", "B04", "B08"]
band_names_20m = ["B05", "B06", "B07", "B11", "B12"]

bands = []
for b in band_names_10m:
    bands.append(decode_band(tile[b]))

from PIL import Image
for b in band_names_20m:
    arr = decode_band(tile[b])  # 534x534
    arr = np.array(Image.fromarray(arr).resize((1068, 1068), Image.BILINEAR))
    bands.append(arr)

s2_l2a = np.stack(bands, axis=0).astype(np.uint16)  # (9, 1068, 1068)

# Day-of-year from timestamp
from datetime import datetime
doy = datetime.strptime(row.timestamp[:8], "%Y%m%d").timetuple().tm_yday

print(f"S2 L2A: {s2_l2a.shape}, DOY: {doy}")

## 4. Predict embeddings

In [ ]:
embedding = model.predict(s2_l2a=s2_l2a, doy=doy)
print(f"Predicted embedding: {embedding.shape}, dtype={embedding.dtype}")
print(f"L2 norm at centre pixel: {np.linalg.norm(embedding[534, 534]):.4f}")

## 5. Visualise: PCA-RGB

Project the 64-dimensional embedding to 3 principal components for visualisation.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def pca_rgb(emb, pca_model=None):
    """Project (H, W, D) embedding to (H, W, 3) RGB via PCA."""
    H, W, D = emb.shape
    flat = emb.reshape(-1, D)
    if pca_model is None:
        pca_model = PCA(n_components=3)
        pca_model.fit(flat)
    rgb = pca_model.transform(flat).reshape(H, W, 3)
    for c in range(3):
        lo, hi = np.percentile(rgb[:, :, c], [2, 98])
        rgb[:, :, c] = np.clip((rgb[:, :, c] - lo) / max(hi - lo, 1e-8), 0, 1)
    return rgb, pca_model

# S2 true-colour for reference
s2_rgb = np.stack([s2_l2a[2], s2_l2a[1], s2_l2a[0]], axis=-1)  # B04, B03, B02
s2_rgb = np.clip(s2_rgb.astype(np.float32) / 3000, 0, 1)

pred_rgb, pca_model = pca_rgb(embedding)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(s2_rgb)
axes[0].set_title("Sentinel-2 RGB")
axes[0].axis("off")
axes[1].imshow(pred_rgb)
axes[1].set_title("BetaEarth Embedding (PCA-RGB)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 6. Compare with real AlphaEarth embedding

Load the real AEF embedding for the same tile directly from [source.coop](https://source.coop/tge-labs/aef) (anonymous S3, no credentials needed).

In [ ]:
import geopandas as gpd
import rasterio as rio
from rasterio.windows import from_bounds
from shapely.geometry import Point
from major_tom_grid import Grid

# Load the AEF spatial index from source.coop (anonymous S3)
aef_df = gpd.read_parquet(
    "s3://us-west-2.opendata.source.coop/tge-labs/aef/v1/annual/aef_index.parquet",
    storage_options={"anon": True},
)

# Convert grid cell to lat/lon
mt_grid = Grid(10, latitude_range=(-90, 90), longitude_range=(-180, 180))
grid_row, grid_col = TARGET_CELL.split("_")
lats, lons = mt_grid.rowcol2latlon([grid_row], [grid_col])

# Spatial lookup → find the COG path
aef_row = aef_df[aef_df.contains(Point(lons[0], lats[0]))].iloc[0]
print(f"AEF COG: {aef_row.path}")

# Read the full tile from source.coop
with rio.Env(AWS_NO_SIGN_REQUEST="YES"):
    with rio.open(aef_row.path) as src:
        aef_emb = src.read().transpose(1, 2, 0).astype(np.float32)  # (H, W, 64)

print(f"AEF embedding: {aef_emb.shape}")

# Per-pixel cosine similarity
dot = (embedding * aef_emb).sum(axis=-1)
pred_norm = np.linalg.norm(embedding, axis=-1) + 1e-8
aef_norm = np.linalg.norm(aef_emb, axis=-1) + 1e-8
cosine_sim = dot / (pred_norm * aef_norm)

print(f"Mean cosine similarity: {cosine_sim.mean():.4f}")
print(f"Std: {cosine_sim.std():.4f}")

In [ ]:
aef_rgb, _ = pca_rgb(aef_emb, pca_model)  # Same PCA for comparable colours

fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
axes[0].imshow(s2_rgb)
axes[0].set_title("S2 RGB")
axes[1].imshow(aef_rgb)
axes[1].set_title("AlphaEarth (PCA-RGB)")
axes[2].imshow(pred_rgb)
axes[2].set_title("BetaEarth (PCA-RGB)")
im = axes[3].imshow(cosine_sim, cmap="RdYlGn", vmin=0.5, vmax=1.0)
axes[3].set_title(f"Cosine sim (mean={cosine_sim.mean():.3f})")
plt.colorbar(im, ax=axes[3], fraction=0.046)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Model variants

BetaEarth provides **7 model variants** with different trade-offs:

| Model | Cos Sim | Params | Key property |
|-------|---------|--------|-------|
| `betaearth-segformer-film` | **0.886** | 0.3M | Best overall |
| `betaearth-segformer-film-hilr` | 0.886 | 0.3M | Alternative frozen variant |
| `betaearth-segformer-film-scratch` | 0.883 | 104.8M | End-to-end trainable |
| `betaearth-segformer` | 0.880 | 104.8M | **No timestamp needed** |
| `betaearth-dinov3-vitl16` | 0.874 | 2.1M | DINOv3 satellite-pretrained |
| `betaearth-dinov3-vits16` | 0.861 | 1.8M | DINOv3 lightweight |
| `betaearth-rgb-only` | 0.836 | 26.3M | **Only 3 RGB bands needed** |

Below we compare two interesting alternatives.

In [ ]:
# RGB-only model — just 3 bands + day-of-year
model_rgb = BetaEarth.from_pretrained("asterisk-labs/betaearth-rgb-only")

s2_rgb_input = s2_l2a[[2, 1, 0]]  # B04, B03, B02 → (3, H, W)
emb_rgb = model_rgb.predict(s2_l1c=s2_rgb_input, doy=doy)

cos_rgb = (emb_rgb * aef_emb).sum(axis=-1) / (np.linalg.norm(emb_rgb, axis=-1) + 1e-8) / aef_norm
print(f"RGB-only mean cosine similarity: {cos_rgb.mean():.4f}")

In [ ]:
# No-FiLM model — no day-of-year needed
model_nofilm = BetaEarth.from_pretrained("asterisk-labs/betaearth-segformer")

emb_nofilm = model_nofilm.predict(s2_l2a=s2_l2a)  # no doy argument

cos_nofilm = (emb_nofilm * aef_emb).sum(axis=-1) / (np.linalg.norm(emb_nofilm, axis=-1) + 1e-8) / aef_norm
print(f"No-FiLM mean cosine similarity: {cos_nofilm.mean():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, cos, title in zip(axes,
    [cosine_sim, cos_nofilm, cos_rgb],
    [f"Full model ({cosine_sim.mean():.3f})",
     f"No FiLM ({cos_nofilm.mean():.3f})",
     f"RGB-only ({cos_rgb.mean():.3f})"]):
    im = ax.imshow(cos, cmap="RdYlGn", vmin=0.5, vmax=1.0)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("Per-pixel cosine similarity with AlphaEarth", fontsize=14)
plt.tight_layout()
plt.show()

## Next steps

- Add **Sentinel-1 and DEM** for higher fidelity: `model.predict(s2_l2a=..., s1=..., dem=..., doy=...)`
- **Average multiple timestamps** for annual embeddings (saturates at ~4 observations)
- Browse all 7 models at [huggingface.co/asterisk-labs](https://huggingface.co/asterisk-labs)
- Code and paper: [github.com/asterisk-labs/beta-earth](https://github.com/asterisk-labs/beta-earth)